In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from src.deep_learning.preprocessing import scale_df, TimeSeriesDataset
from src.deep_learning.finetuning import objective
import joblib
import optuna
from functools import partial

In [ ]:
random_seed = 42
np.random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
train_df = pd.read_parquet(r"..\data\processed\train.parquet")
val_df = pd.read_parquet(r"..\data\processed\val.parquet")
test_df = pd.read_parquet(r"..\data\processed\test.parquet")
consumption_df = pd.read_parquet(r"..\data\processed\consumption_fe.parquet")

In [ ]:
train_df.head(1)

,date,consumption,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos
0,2018-01-01,27412.81,0.0,1.0,0.0,1.0,0.5,0.866025


In [ ]:
feature_cols = ["consumption",
    "hour_sin", "hour_cos",
    "day_sin", "day_cos",
    "month_sin", "month_cos"
]

target = "consumption"

scale_cols = ["consumption"]

In [ ]:
scaler = StandardScaler()

scaler_path = r"..\models\scaler_ft.pkl"

train_scaled = scale_df(df=train_df, train=True, scaler=scaler, scale_cols=scale_cols)

joblib.dump(scaler, scaler_path)

val_scaled = scale_df(df=val_df, train=False, scaler=scaler, scale_cols=scale_cols)

test_scaled = scale_df(df=test_df, train=False, scaler=scaler, scale_cols=scale_cols)

In [ ]:
train_scaled.head(2)

,date,consumption,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos
0,2018-01-01 00:00:00,-1.350096,0.000000,1.000000,0.0,1.0,0.5,0.866025
1,2018-01-01 01:00:00,-1.539594,0.258819,0.965926,0.0,1.0,0.5,0.866025


In [ ]:
window_size = 168
horizon = 24
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_dataset = TimeSeriesDataset(df=train_scaled, feature_cols=feature_cols, target_col=target, window_size=window_size, horizon=horizon) 
val_dataset = TimeSeriesDataset(df=val_scaled, feature_cols=feature_cols, target_col=target, window_size=window_size, horizon=horizon) 
test_dataset = TimeSeriesDataset(df=test_scaled, feature_cols=feature_cols, target_col=target, window_size=window_size, horizon=horizon) 

In [ ]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
study = optuna.create_study(
    study_name="lstm_finetuning",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=random_seed),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

[I 2026-02-11 12:56:13,687] A new study created in memory with name: lstm_finetuning


In [ ]:
objective_fn = partial(objective,
                        train_loader=train_loader,
                        val_loader=val_loader,
                        input_size=len(feature_cols),
                        horizon=horizon,
                        device=device,
                        num_epochs=50)

In [ ]:
study.optimize(objective_fn, n_trials=50)

print("Best validation loss:", study.best_trial.value)
print("Best hyperparameters:")
print(study.best_trial.params)

[I 2026-02-11 12:58:06,662] Trial 0 finished with value: 0.023707333226899226 and parameters: {'hidden_size': 96, 'num_layers': 3, 'dropout': 0.36599697090570255, 'lin_features': 128, 'lr': 0.0001432249371823026, 'weight_decay': 4.207053950287936e-06}. Best is trial 0 with value: 0.023707333226899226.
[I 2026-02-11 12:58:37,101] Trial 1 finished with value: 0.6543192066469862 and parameters: {'hidden_size': 32, 'num_layers': 3, 'dropout': 0.3005575058716044, 'lin_features': 144, 'lr': 0.00010485387725194633, 'weight_decay': 0.00757947995334801}. Best is trial 0 with value: 0.023707333226899226.
[I 2026-02-11 13:05:48,769] Trial 2 finished with value: 0.023477978933284797 and parameters: {'hidden_size': 224, 'num_layers': 1, 'dropout': 0.09091248360355031, 'lin_features': 48, 'lr': 0.0002014847788415866, 'weight_decay': 0.0001256104370001356}. Best is trial 2 with value: 0.023477978933284797.
[I 2026-02-11 13:06:52,812] Trial 3 finished with value: 0.024292064073073855 and parameters: {

Best validation loss: 0.017443829425781304
Best hyperparameters:
{'hidden_size': 224, 'num_layers': 2, 'dropout': 0.17498226896177801, 'lin_features': 32, 'lr': 0.0006477616087737383, 'weight_decay': 3.680266245751368e-06}
